# ATC controller fine-tune: end to end

In [ ]:
!pip install -q -U curatorkit aligntune auditkit datasets

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, torch
os.makedirs("/content/drive/MyDrive/atc_tune/dataset/sft", exist_ok=True)

try:
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped:", e)

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
import json, os, re
from collections import Counter
from datasets import load_dataset

# ---- single place for every path in the pipeline ----
try:
    import google.colab  # noqa: F401
    BASE = "/content/drive/MyDrive/atc_tune"
except ImportError:
    BASE = "."

DATASET_DIR     = f"{BASE}/dataset/sft"                # put the two .parquet files here
DATA_DIR        = f"{BASE}/data"
CLEAN_TRAIN     = f"{DATA_DIR}/train.jsonl"            # cleaned, pre-curation
CLEAN_TEST      = f"{DATA_DIR}/test.jsonl"             # cleaned test set (used for eval)
CURATED_DIR     = f"{DATA_DIR}/curated"
CURATED_TRAIN   = f"{CURATED_DIR}/sft_messages.jsonl"  # written by curatorkit (messages format)
FINAL_TRAIN     = f"{DATA_DIR}/final/train_final.jsonl"  # exact file the trainer reads
OUTPUT_DIR      = f"{BASE}/outputs/aya-ground-controller"
os.makedirs(f"{DATA_DIR}/final", exist_ok=True)

## 1. Load

In [ ]:
ds = load_dataset(
    "parquet",
    data_files={
        "train": f"{DATASET_DIR}/train-00000-of-00001.parquet",
        "test":  f"{DATASET_DIR}/test-00000-of-00001.parquet",
    },
)
print(ds)

## 2. Explore (scenarios and controller tags)

In [ ]:
def get_scenario(system_text):
    m = re.search(r"Scenario:\s*([^\.\n]+)", system_text, re.I)
    return m.group(1).strip().lower() if m else "unknown"

scenarios = Counter()
scenario_controller = {}

for row in ds["train"]:
    msgs = row["messages"]
    system = msgs[0]["content"] if msgs and msgs[0]["role"] == "system" else ""
    sc = get_scenario(system)
    scenarios[sc] += 1
    tags = [
        t.group(1)
        for m in msgs if m["role"] == "assistant"
        for t in [re.match(r"\[([A-Z_]+)\]", m["content"].strip())] if t
    ]
    scenario_controller.setdefault(sc, Counter()).update(tags)

for sc, n in scenarios.most_common():
    print(f"{n:5} | {sc:32} | {dict(scenario_controller[sc])}")

 1637 | real atc uwb                     | {'ATC': 4084}
  447 | pattern work                     | {'ATC_GROUND': 447, 'ATC_TOWER': 1647}
  366 | vfr departure towered            | {'ATC_GROUND': 366, 'ATC_TOWER': 835}
  356 | vfr arrival towered              | {'ATC_APPROACH': 459, 'ATC_TOWER': 1068}
  354 | ctaf nontowered                  | {}
  314 | vfr flight following             | {'ATC_APPROACH': 1405}
  225 | go around                        | {'ATC_TOWER': 675}
  222 | weather diversion                | {'ATC_APPROACH': 798}
  218 | class bravo transition           | {'ATC_APPROACH': 1090}
  189 | pirep                            | {'ATC_CENTER': 378}
  180 | progressive taxi                 | {'ATC_GROUND': 540}
  168 | real atc atco2                   | {'ATC': 193}
   42 | emergency engine rough           | {'ATC_APPROACH': 126}
   41 | emergency electrical             | {'ATC_APPROACH': 82}
   37 | emergency passenger medical      | {'ATC_APPROACH': 74}
   34 | emergenc

## 3. Clean
`normalize_message` strips the `[ATC_*]` / `[PILOT]` tags. `repair_conversation` then fixes the structural problems that
caused the 4,498 "invalid conversations" in the old validation (leading assistant turns, trailing user turns,
back-to-back turns with the same role). Every drop is counted by reason.



In [ ]:
KEEP_SCENARIOS = None   # e.g. {"progressive taxi", "pattern work"}
KEEP_TAGS      = None   # e.g. {"ATC_GROUND"}  (checked BEFORE tags are stripped)

CONTROLLER_PREFIXES = ["[ATC_GROUND]", "[ATC_TOWER]", "[ATC_APPROACH]", "[ATC]"]

def normalize_message(msg):
    role = (msg.get("role") or "").strip().lower()
    content = str(msg.get("content") or "").strip()
    if role == "assistant":
        for p in CONTROLLER_PREFIXES:
            if content.startswith(p):
                content = content[len(p):].strip()
                break
    elif role == "user" and content.startswith("[PILOT]"):
        content = content[len("[PILOT]"):].strip()
    return {"role": role, "content": content}


def repair_conversation(messages):
    """Return (clean_messages | None, drop_reason | None)."""
    msgs = [normalize_message(m) for m in messages]
    msgs = [m for m in msgs if m["content"] and m["role"] in {"system", "user", "assistant"}]

    system = [m for m in msgs if m["role"] == "system"][:1]
    turns  = [m for m in msgs if m["role"] != "system"]

    # merge consecutive same-role turns
    merged = []
    for m in turns:
        if merged and merged[-1]["role"] == m["role"]:
            merged[-1]["content"] += " " + m["content"]
        else:
            merged.append(dict(m))

    while merged and merged[0]["role"] != "user":        # must start with user
        merged.pop(0)
    while merged and merged[-1]["role"] != "assistant":  # must end with assistant
        merged.pop()

    if len(merged) < 2:
        return None, "no_user_assistant_pair"
    return system + merged, None


def prepare(example, stats):
    msgs = example["messages"]
    if not msgs:
        stats["empty"] += 1
        return None

    system_text = msgs[0]["content"] if msgs[0]["role"] == "system" else ""
    if KEEP_SCENARIOS and get_scenario(system_text) not in KEEP_SCENARIOS:
        stats["filtered_scenario"] += 1
        return None
    if KEEP_TAGS:
        tags = {t.group(1) for m in msgs if m["role"] == "assistant"
                for t in [re.match(r"\[([A-Z_]+)\]", m["content"].strip())] if t}
        if not tags & KEEP_TAGS:
            stats["filtered_tag"] += 1
            return None

    cleaned, reason = repair_conversation(msgs)
    if reason:
        stats[reason] += 1
        return None
    return {"messages": cleaned}   # no id: training only needs the messages


def build_split(split):
    stats, rows = Counter(), []
    for ex in ds[split]:
        item = prepare(ex, stats)
        if item:
            rows.append(item)
    print(f"{split}: kept {len(rows)} / {len(ds[split])} | dropped: {dict(stats)}")
    return rows

train_rows = build_split("train")
test_rows  = build_split("test")

## 4. Validate

In [ ]:
def validate_dataset(rows, name):
    reasons = Counter()
    for row in rows:
        roles = [m["role"] for m in row["messages"]]
        if roles and roles[0] == "system":
            roles = roles[1:]
        if not roles:
            reasons["empty"] += 1
        elif roles[0] != "user":
            reasons["starts_not_user"] += 1
        elif roles[-1] != "assistant":
            reasons["ends_not_assistant"] += 1
        elif any(roles[i] == roles[i-1] for i in range(1, len(roles))):
            reasons["repeated_role"] += 1
    print(f"{name}: invalid = {sum(reasons.values())} {dict(reasons)}")
    assert not reasons, f"{name} still has invalid conversations"

validate_dataset(train_rows, "train")
validate_dataset(test_rows, "test")

## 5. Save the final cleaned data (before curation and training)

In [ ]:
def save_jsonl(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"saved {len(rows):5} rows -> {path}")

save_jsonl(train_rows, CLEAN_TRAIN)
save_jsonl(test_rows,  CLEAN_TEST)

with open(f"{DATA_DIR}/clean_manifest.json", "w") as f:
    json.dump({
        "train_rows": len(train_rows), "test_rows": len(test_rows),
        "keep_scenarios": sorted(KEEP_SCENARIOS) if KEEP_SCENARIOS else None,
        "keep_tags": sorted(KEEP_TAGS) if KEEP_TAGS else None,
    }, f, indent=2)

saved  3902 rows -> /content/drive/MyDrive/atc_tune/data/train.jsonl
saved   427 rows -> /content/drive/MyDrive/atc_tune/data/test.jsonl


## 6. Curate (dedup + clean)
Exports `messages` format so the output has the `messages` column the trainer expects
(`sharegpt` writes `conversations` with `from`/`value`, which doesn't match `messages_column="messages"`).

In [ ]:
from curatorkit import Curator, CuratorConfig

result = Curator(
    CuratorConfig(
        dataset={"name": CLEAN_TRAIN},
        dedup="minhash",
        clean=True,
        export_formats=["messages"],
        output_dir=CURATED_DIR,
    )
).run()

result.print_summary()
assert os.path.exists(CURATED_TRAIN), f"curator did not write {CURATED_TRAIN}"

## 7. Final training file (re-validated after curation)

In [ ]:
with open(CURATED_TRAIN, encoding="utf-8") as f:
    curated_rows = [json.loads(line) for line in f if line.strip()]

validate_dataset(curated_rows, "curated train")
print(f"clean {len(train_rows)} -> curated {len(curated_rows)} "
      f"({len(train_rows) - len(curated_rows)} removed by dedup/clean)")

save_jsonl(curated_rows, FINAL_TRAIN)   # <- exactly what the trainer reads

## 8. Train

In [ ]:
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets curatorkit auditkit


In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=FINAL_TRAIN, split="train")
train_ds = train_ds.select_columns(["messages"])
print(train_ds)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

#load model and the tokenizer

MODEL = "CohereLabs/tiny-aya-global"

# T4 (compute capability 7.5) has no real bf16, so pick the dtype from the capability
# instead of torch.cuda.is_bf16_supported(), which can wrongly return True on a T4.
use_bf16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if use_bf16 else torch.float16
print("compute dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=1024,                 # use 2048 if memory allows (older TRL calls this max_seq_length)
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_steps=100,                  # checkpoints go to Drive
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,          # "messages" column is handled automatically
    processing_class=tokenizer,
    peft_config=lora,
)

from collections import Counter
print(Counter((str(p.dtype), p.requires_grad) for p in trainer.model.parameters()))

In [ ]:
import torch
from collections import Counter

m = trainer.model
print("args  fp16:", trainer.args.fp16, "| bf16:", trainer.args.bf16)
print(Counter((str(p.dtype), p.requires_grad) for p in m.parameters()))

bf = [n for n, p in m.named_parameters() if p.dtype == torch.bfloat16]
print(len(bf), bf[:15])

args  fp16: True | bf16: False
Counter({('torch.bfloat16', True): 504, ('torch.uint8', False): 252, ('torch.float32', False): 38})
504 ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.w

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

cast to fp32: 0


Step,Training Loss
10,1.500907
20,0.621799
30,0.433580
40,0.314924
50,0.294858
60,0.248998
70,0.229502
80,0.210608
90,0.203968
100,0.193438


Saved to /content/drive/MyDrive/atc_tune/outputs/aya-ground-controller


## 9. Evaluate
Each eval sample carries the **same system prompt the model was trained with** plus the conversation so far, and the
target is the next controller reply.

In [ ]:
%pip install auditkit

In [ ]:
import torch
import auditkit as ak
from transformers import AutoTokenizer, AutoModelForCausalLM

def build_eval_samples(path, max_samples=None):
    samples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            msgs = json.loads(line)["messages"]
            # first assistant reply, with everything before it as context
            idx = next((i for i, m in enumerate(msgs) if m["role"] == "assistant"), None)
            if idx is None:
                continue
            samples.append(ak.Sample(
                input=json.dumps(msgs[:idx], ensure_ascii=False),  # context as JSON string
                target=msgs[idx]["content"],
            ))
            if max_samples and len(samples) >= max_samples:
                break
    return samples

samples = build_eval_samples(CLEAN_TEST)
print("Evaluation samples:", len(samples))

Evaluation samples: 427


In [ ]:
%pip install --upgrade torchao transformers
%pip install -U bitsandbytes>=0.46.1

In [ ]:
import sys
import site
import importlib

# Refresh active site-packages directory path
importlib.reload(site)
for path in site.getsitepackages():
    if path not in sys.path:
        sys.path.insert(0, path)

try:
    import bitsandbytes
    importlib.reload(bitsandbytes)
except Exception:
    pass

import json
import torch
import auditkit as ak
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_PATH = OUTPUT_DIR

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure 4-bit quantization to prevent CPU offloading on the T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)

def generate_responses(prompts):
    """Handles both a single prompt and a batch list of prompts from auditkit."""
    is_single = isinstance(prompts, str)
    prompt_list = [prompts] if is_single else prompts
    outputs = []

    for p in prompt_list:
        text = p if isinstance(p, str) else getattr(p, "input", str(p))

        try:
            messages = json.loads(text)
            assert isinstance(messages, list)
        except (json.JSONDecodeError, AssertionError, TypeError):
            messages = [{"role": "user", "content": text}]

        encoded = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt"
        )

        # Safely extract the tensor if encoded is a BatchEncoding dict
        if isinstance(encoded, dict) or hasattr(encoded, "keys"):
            input_ids = encoded["input_ids"].to(model.device)
        else:
            input_ids = encoded.to(model.device)

        attention_mask = torch.ones_like(input_ids).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=50,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response_text = tokenizer.decode(
            generated[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        ).strip()
        outputs.append(response_text)

    return outputs[0] if is_single else outputs

# Run evaluation on the first 10 samples to verify speed and correctness first
result = ak.evaluate(
    samples[:10],
    model=generate_responses,
    scorers=[ak.F1Score()],
    experiment_name="aya-ground-controller",
    cache=False,
)
print(result.headline)

## 10. Spot checks (use a system prompt in the training format)

In [ ]:
SYSTEM = ("You are simulating ATC communications at Houston Hobby (KHOU), a Class B airport in the Gulf Coast region. "
          "Aircraft: Cessna 172, Callsign: N4273A. Scenario: progressive taxi. "
          "Use standard FAA phraseology and readback/hearback procedures.")

def ask_controller(pilot_call):
    ctx = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": pilot_call.lower()}]
    return generate_responses([json.dumps(ctx)])[0]

for t in [
    "Hobby Ground, Skyhawk November 4273 Alpha, at the ramp, request taxi runway 17.",
    "Ground, November 4273 Alpha, holding short runway 17, request crossing.",
    "Ground, Skyhawk 4273 Alpha, unfamiliar with airport, request progressive taxi.",
    "Ground, November 4273 Alpha, ready to taxi.",
]:
    print("\nPILOT:", t)
    print("ATC:  ", ask_controller(t))

### Improved Spot-Check with Decoding Adjustments
To address the phrase-merging and loop artifacts, we introduce generation penalties (`repetition_penalty=1.15`, `no_repeat_ngram_size=4`) and switch to sampling to encourage cleaner terminations.

In [ ]:
def generate_responses_improved(prompts):
    is_single = isinstance(prompts, str)
    prompt_list = [prompts] if is_single else prompts
    outputs = []

    for p in prompt_list:
        text = p if isinstance(p, str) else getattr(p, "input", str(p))

        try:
            messages = json.loads(text)
            assert isinstance(messages, list)
        except (json.JSONDecodeError, AssertionError, TypeError):
            messages = [{"role": "user", "content": text}]

        encoded = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt"
        )

        if isinstance(encoded, dict) or hasattr(encoded, "keys"):
            input_ids = encoded["input_ids"].to(model.device)
        else:
            input_ids = encoded.to(model.device)

        attention_mask = torch.ones_like(input_ids).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=60,
                do_sample=True, # Switch to sampling to help avoid repetitive traps
                temperature=0.4, 
                top_p=0.85,
                repetition_penalty=1.15, # Penalizes exact repetition
                no_repeat_ngram_size=4, # Prevents repetitive 4-token cycles
                pad_token_id=tokenizer.eos_token_id
            )

        response_text = tokenizer.decode(
            generated[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        ).strip()
        outputs.append(response_text)

    return outputs[0] if is_single else outputs

def ask_controller_improved(pilot_call):
    ctx = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": pilot_call.lower()}]
    return generate_responses_improved([json.dumps(ctx)])[0]


for t in [
    "Hobby Ground, Skyhawk November 4273 Alpha, at the ramp, request taxi runway 17.",
    "Ground, November 4273 Alpha, holding short runway 17, request crossing.",
    "Ground, Skyhawk 4273 Alpha, unfamiliar with airport, request progressive taxi.",
    "Ground, November 4273 Alpha, ready to taxi.",
]:
    print("\nPILOT:", t)
    print("ATC:  ", ask_controller_improved(t))